

#  Colab 1

В этом блокноте мы реализуем полноценный **ПАЙПЛАЙН** для **выучивания эмбеддингов вершин**. Нам предстоит выполнить следующие 3 шага.

Для начала загрузим ставший классикой в сетевом анализе граф - [Karate Club Network](https://en.wikipedia.org/wiki/Zachary%27s_karate_club). И исследуем несколько графовых статистик для сети социальных связей каратистов.

Затем - преобразуем граф в тензор PyTorch tensor, что позволит заняться машинным обучением на графе.

Наконец, мы завершим первый алгоритм машинного обучения на графах: модель эмбеддингов вершин. Наша модель проще, чем алгоритмы DeepWalk / node2vec, рассмотренные в лекции. Однако реализация с помощью PyTorch будет интересным и непростым приключением.



**Обратите внимание**:  
Убедитесь в том, чтобы **последовательно запускать все ячейки с кодом**, таким образом промежуточные переменные будут переданы в последующую ячейку

## 1 Графы - основы
Для начала загрузим ставший классикой в сетевом анализе граф - [Karate Club Network](https://en.wikipedia.org/wiki/Zachary%27s_karate_club). И исследуем несколько графовых статистик для сети социальных связей каратистов.

## Установка
Мы будем много раз пользоваться NetworkX в этом блокноте.

In [ ]:
import networkx as nx

## Сети социальных связей каратистов Захари

[Karate Club Network](https://en.wikipedia.org/wiki/Zachary%27s_karate_club) -  это граф, описывающий социальную сеть 34 членов клуба карате и документирующий связи между учасниками, взаимодействовавшими вне клуба.

In [ ]:
G = nx.karate_club_graph()

# G - неориентированный граф
type(G)

In [ ]:
# Нарисуем граф
nx.draw(G, with_labels = True)

## Вопрос 1: Какова средняя степень вершины в сети связей каратистов?

In [ ]:
def average_degree(num_edges, num_nodes):
  # ЗАДАЧА: Реализуйте функцию, принимающую количество рёбер
  # и количество вершин, и возвращающую среднюю степень в
  # графе. Округлите результат до ближайшего целого числа
  # (например, 3.3 → 3, а 3.7 → 4)

  avg_degree = 0

  ############# Ваш код здесь #############

  #########################################

  return avg_degree

num_edges = G.number_of_edges()
num_nodes = G.number_of_nodes()
avg_degree = average_degree(num_edges, num_nodes)
print("Average degree of karate club network is {}".format(avg_degree))

## Вопрос 2: Каков средний коэффициент кластеризации в сети связей каратистов?

In [ ]:
def average_clustering_coefficient(G):
  # ЗАДАЧА: Реализуйте функцию, которая принимает nx.Graph
  # и возвращает средний коэффициент кластеризации. Округлите
  # дробную часть результата до 2 знаков
  # (например, 3.333 → 3.33, а 3.7571 → 3.76)

  avg_cluster_coef = 0

  ############# Ваш код здесь #############
  ## Обратите внимание:
  ## 1: Пожалуйста используйте функцию кластеризации NetworkX

  #########################################

  return avg_cluster_coef

avg_cluster_coef = average_clustering_coefficient(G)
print("Average clustering coefficient of karate club network is {}".format(avg_cluster_coef))

## Вопрос 3: Чему равно значение PageRank для вершины 0 (вершина с id 0) после одной итерации?

Пожалуйста выполните задание путём реализации уравнения PageRank:  
    
$r_j = \sum_{i \rightarrow j} \beta \frac{r_i}{d_i} + (1 - \beta) \frac{1}{N}$

In [ ]:
def one_iter_pagerank(G, beta, r0, node_id):
  # ЗАДАНИЕ: Реализуйте функцию, принимающую nx.Graph, beta, r0 и id вершины.
  # Возвращаемое значение r1 - это результат одной итерации PageRank для указанной вершины.
  # Пожалуйста округлите r1 до двух десятичных знаков.

  r1 = 0

  ############# Ваш код здесь #############
  ## Обратите внимание:
  ## 1: You should not use nx.pagerank

  #########################################

  return r1

beta = 0.8
r0 = 1 / G.number_of_nodes()
node = 0
r1 = one_iter_pagerank(G, beta, r0, node)
print("The PageRank value for node 0 after one iteration is {}".format(r1))

## Вопрос 4: Чему равно значение (ненормированное) степени близости (closeness centrality) для вершины 5 в сети клуба каратистов?


Уравнение для степени близости:  
$c(v) = \frac{1}{\sum_{u \neq v}\text{длина кратчайшего пути между } u \text{ и } v}$

In [ ]:
def closeness_centrality(G, node=5):
  # ЗАДАЧА: Реализуйте функцию, вычисляющую степень близости для
  # вершины в сети клуба каратистов. G - это граф сети связей клуба
  # каратистов, а node - id вершины в графе. Пожалуйста округлите
  # значение степени близости до 2 десятичных знаков.

  closeness = 0

  ############# Ваш код здесь #############
  ## Обратите внимание:
  ## 1: Вы можете воспользоваться функцией networkx
  ## 2: Функция networkx возвращает нормированную степень близости, которая
  ## отличается от ненормированной, рассмотренной в лекции.

  #########################################

  return closeness

node = 5
closeness = closeness_centrality(G, node=node)
print("The karate club network has closeness centrality {}".format(closeness))

# 2 Граф в Тензор

Затем - преобразуем граф $G$ в тензор PyTorch, что позволит заняться машинным обучением на графе.

## Установка
Проверим корректность установки PyTorch

In [ ]:
import torch
print(torch.__version__)

1.8.0+cu101


## PyTorch - основы тензоров

Мы можем сгенерировать тензор PyTorch, заполненный нулями, единицами, либо псевдослучайными значениями.

In [ ]:
# Инициализируем тензор размерности 3 x 4, заполненный единицами
ones = torch.ones(3, 4)
print(ones)

# Тензор размерности 3 x 4, заполненный нулями
zeros = torch.zeros(3, 4)
print(zeros)

# Тензор размерности 3 x 4, заполненный псевдослучайными значениями из интервала [0, 1)
random_tensor = torch.rand(3, 4)
print(random_tensor)

# Посмотрим размерность тензора
print(ones.shape)

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])
tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
tensor([[0.1504, 0.6981, 0.8470, 0.2549],
        [0.3369, 0.8038, 0.9007, 0.6615],
        [0.4338, 0.6560, 0.0322, 0.1211]])
torch.Size([3, 4])


Тензор PyTorch содержит элементы одного типа, заданного параметром `dtype`.

In [ ]:
# Создадим тензор размерности 3 x 4, заполненный 32-битными нулями с плавающей точкой
zeros = torch.zeros(3, 4, dtype=torch.float32)
print(zeros.dtype)

# Сменим тип данных (dtype) на 64-битный целочисленный
zeros = zeros.type(torch.long)
print(zeros.dtype)

torch.float32
torch.int64


## Вопрос 5: Возьмём список рёбер графа связей клуба каратистов и преобразуем в `torch.LongTensor`. Чему равно значение `torch.sum` для тензора `pos_edge_index`?


In [ ]:
def graph_to_edge_list(G):
  # ЗАДАЧА: Реализуйте функцию, которая принимает nx.Graph и возвращает список рёбер.
  # Возвращаемый edge_list представляет собой список кортежей, в котором
  # каждая пара значений представляет собой ребро, соединяющее две
  # смежные вершины.

  edge_list = []

  ############# Ваш код здесь #############

  #########################################

  return edge_list

def edge_list_to_tensor(edge_list):
  # ЗАДАЧА: Реализуйте функцию, которая преобразует edge_list в тензор.
  # Подаваемый на вход edge_list представляет собой список кортежей, а
  # тензор на выходе должен быть размерности [2 x len(edge_list)].

  edge_index = torch.tensor([])

  ############# Ваш код здесь #############

  #########################################

  return edge_index

pos_edge_list = graph_to_edge_list(G)
pos_edge_index = edge_list_to_tensor(pos_edge_list)
print("The pos_edge_index tensor has shape {}".format(pos_edge_index.shape))
print("The pos_edge_index tensor has sum value {}".format(torch.sum(pos_edge_index)))

## Вопрос 6: Пожалуйста реализуйте негативный семплинг. Затем определите, какие рёбра (от edge_1 до edge_5) могут быть негативными в графе клуба каратистов?

In [ ]:
import random

def sample_negative_edges(G, num_neg_samples):
  # ЗАДАЧА: Реализуйте функцию, возвращающую список негативных (отсутствующих) рёбер.
  # Количество рёбер задаёт параметр num_neg_samples. Вам нет нужды учитывать
  # граничный случай, когда количество возможных негативных рёбер менее чем
  # num_neg_samples. Достаточно, чтобы функция работала на графе сети связей
  # картистов. В данной реализации петли (рёбра вида (v,v)) не должны считаться
  # ни позитивными, ни негативными. Также обратите внимание на то, что граф сети
  # связей клуба каратистов - неориентированный граф, если (0, 1) - присутствует,
  # как думаете, будет ли (1, 0) отсутствовать?

  neg_edge_list = []

  ############# Ваш код здесь #############

  #########################################

  return neg_edge_list

# Выберем 78 негативных рёбер
neg_edge_list = sample_negative_edges(G, len(pos_edge_list))

# Преобразуем список негативных рёбер в тензор
neg_edge_index = edge_list_to_tensor(neg_edge_list)
print("The neg_edge_index tensor has shape {}".format(neg_edge_index.shape))

# Какие из следующих рёбер могут быть негативными?
edge_1 = (7, 1)
edge_2 = (1, 33)
edge_3 = (33, 22)
edge_4 = (0, 4)
edge_5 = (4, 2)

############# Ваш код здесь #############
## Обратите внимание:
## 1: Для каждого из 5 рёбер, определите, может ли оно быть негативным

#########################################

# 3 Выучим эмбеддинги вершин
Наконец, мы завершим первый алгоритм машинного обучения на графах: модель эмбеддингов вершин.


## Установка

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

print(torch.__version__)

Для создания наших собственных методов выучивания эмбеддингов мы будем много раз обращаться к модулю [`nn.Embedding`](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html) PyTorch. Посмотрим, как использовать `nn.Embedding`:

In [ ]:
# Инициализируем слой эмбеддингов
# Предположим, что мы хотим создать эмбеддинги для 4 элементов (например, вершин)
# Каждый элемент будет представлен 8-мерным вектором

emb_sample = nn.Embedding(num_embeddings=4, embedding_dim=8)
print('Sample embedding layer: {}'.format(emb_sample))

Мы можем выбрать элементы из матрицы эмбеддингов, используя индексы тензора

In [ ]:
# Выберем эмбеддинг из emb_sample
id = torch.LongTensor([1])
print(emb_sample(id))

# Выберем несколько эмбеддингов
ids = torch.LongTensor([1, 3])
print(emb_sample(ids))

# Посмотрим размерность матрицы весов эмбеддинга
shape = emb_sample.weight.data.shape
print(shape)

# Перезапишем веса тензором из единиц
emb_sample.weight.data = torch.ones(shape)

# Проверим инициализацию эмбеддингов
ids = torch.LongTensor([0, 3])
print(emb_sample(ids))

Пришло время создать матрицу эмбеддингов вершин графа!
- Мы хотим назначить **16-разрядный** вектор для каждой вершины в графе связей каратистов.
- Мы также хотим инициализировать матрицу значениями, равномерно распределёнными на интервале $[0, 1)$. Рекомендуем воспользоваться [`torch.rand`](https://pytorch.org/docs/stable/generated/torch.rand.html).

In [ ]:
# Пожалуйста не изменяйте и не сбрасывайте значение random seed
torch.manual_seed(1)

def create_node_emb(num_node=34, embedding_dim=16):
  # ЗАДАЧА: Реализуйте функцию, создающую матрицу эмбеддингов вершин.
  # Возвращающу слой torch.nn.Embedding. Вам не требуется изменять
  # значения num_node и embedding_dim. Матрица весов возвращаемого
  # слоя должна быть инизиализирована из равномерного распределения.

  emb = None

  ############# Ваш код здесь #############

  #########################################

  return emb

emb = create_node_emb()
ids = torch.LongTensor([0, 3])

# Выведем на экраан слой эмбеддингов
print("Embedding: {}".format(emb))

# Пример обращения к эмбеддингам вершин 0 и 3
print(emb(ids))

## Визуализируем эмбеддинги в их начальном состоянии

Один из путей к понимаю матрицы эмбеддингов - визуализировать её содержимое на плоскости.

Здесь мы реализуем функцию визуализации для вас.
Сначала проведём анализ принципиальных компонент для снижения размерности до 2D.
Затем отрисуем каждую точку, окрашенную соответственно принадлежности сообществу.

In [ ]:
def visualize_emb(emb):
  X = emb.weight.data.numpy()
  pca = PCA(n_components=2)
  components = pca.fit_transform(X)
  plt.figure(figsize=(6, 6))
  club1_x = []
  club1_y = []
  club2_x = []
  club2_y = []
  for node in G.nodes(data=True):
    if node[1]['club'] == 'Mr. Hi':
      club1_x.append(components[node[0]][0])
      club1_y.append(components[node[0]][1])
    else:
      club2_x.append(components[node[0]][0])
      club2_y.append(components[node[0]][1])
  plt.scatter(club1_x, club1_y, color="red", label="Mr. Hi")
  plt.scatter(club2_x, club2_y, color="blue", label="Officer")
  plt.legend()
  plt.show()

# Визуализируем эмбеддинги в их начальном (псевдослучайном) состоянии
Visualize the initial random embeddding
visualize_emb(emb)

## Вопрос 7: Выучим эмбеддинги! Какого наилучшего качества классификации удалось достигнуть? Чему равно значение функции потерь (loss) и доля верных ответов (accuracy)?

In [ ]:
from torch.optim import SGD

def accuracy(pred, label):
  # ЗАДАЧА: Реализуйте функцию, подсчитывающую долю верных ответов (accuracy).
  # Эта функция принимает тензор с предсказаниями (на выходе после сигмоиды) и
  # тензор с метками классов (torch.LongTensor). Предсказанные значения более 0.5
  # классифицируются как класс 1. В противном случае - как класс 0.
  # Возвращаемое значение следует округлить до 4 десятичных позиций.
  # Например, доля верных ответов 0.82956 → 0.8296.

  accu = 0.0

  ############# Ваш код здесь #############

  #########################################

  return accu

def train(emb, loss_fn, sigmoid, train_label, train_edge):
  # ЗАДАЧА: Обучите слой эмбеддингов. Вы можете изменять количество эпох и
  # скорость обучения. В общем, вам следует реализовать следующее:
  # (1) Взять эмбеддинги вершин из train_edge
  # (2) Скалярно умножить эмбеддинги для каждой пары вершин
  # (3) Отправить результат в сигмоиду
  # (4) Направить выход сигмоиды в loss_fn
  # (5) Вывести на экран значение функции потерь и долю верных ответов эпохи
  # (для самопроверки, значение функции потерь должно снижаться по ходу обучения)

  epochs = 500
  learning_rate = 0.1

  optimizer = SGD(emb.parameters(), lr=learning_rate, momentum=0.9)

  for i in range(epochs):

    ############# Ваш код здесь #############

    #########################################

loss_fn = nn.BCELoss()
sigmoid = nn.Sigmoid()

# Создадим позитивные и негативные метки классо
pos_label = torch.ones(pos_edge_index.shape[1], )
neg_label = torch.zeros(neg_edge_index.shape[1], )

# Конкатенируем позитивне и негативные метки в один тензор
train_label = torch.cat([pos_label, neg_label], dim=0)

# Конкатенируем позитивные и негативные рёбра в один тензор
# Поскольку граф весьма мал, мы не будем разбивать данные на наборы для валидации/теста
train_edge = torch.cat([pos_edge_index, neg_edge_index], dim=1)

train(emb, loss_fn, sigmoid, train_label, train_edge)

## Визуализируем полученные эмбеддинги

Здесь мы и нарисуем результат обучения!

Вы можете сравнить полученную диаграмму с предыдущим изображением эмбеддингов.
После обучения должно быть видно, что два класса разделены более очевидно.

Это - также отличная проверка вашей реализации.

In [ ]:
# Визуализируем обученные эмбеддинги
visualize_emb(emb)

# Ответы

Направить ответы по ссылке: https://forms.gle/pZdWAxpVV3rqn7QPA